
## 📦 Install Dependencies

In [ ]:
!pip install -q accelerate bitsandbytes datasets huggingface_hub peft scikit-learn transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 517.2/517.2 kB 37.5 MB/s eta 0:00:00


## 📚 Libraries

In [ ]:
from collections import Counter
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login
import numpy as np, os, random, torch
from torch.nn import CrossEntropyLoss
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, PeftModel

🔕 Disable Weights & Biases

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"

## 🔐 Login to Hugging Face Hub

In [ ]:
hf_token = os.environ.get('HF_Token') or userdata.get('HF_Token')

if hf_token:
    login(token=hf_token)
    print("HuggingFace login successful.")
else:
    print("HuggingFace token not found. Please set the HF_TOKEN environment variable or store it in Colab secrets.")

HuggingFace login successful.


In [ ]:
dataset = load_dataset("dair-ai/emotion")

emotion_map = {
    0: "sadness", 1: "joy", 2: "love",
    3: "anger", 4: "fear", 5: "surprise"
}

def convert_label(ex):
    ex["emotion"] = emotion_map[ex["label"]]
    return ex

dataset["train"] = dataset["train"].map(convert_label)
dataset["test"]  = dataset["test"].map(convert_label)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## 📥 Load dair-ai/emotion Dataset

In [ ]:
dataset = load_dataset("dair-ai/emotion")

emotion_map = {
    0: "sadness", 1: "joy", 2: "love",
    3: "anger", 4: "fear", 5: "surprise"
}

def convert_label(ex):
    ex["emotion"] = emotion_map[ex["label"]]
    return ex

dataset["train"] = dataset["train"].map(convert_label)
dataset["test"]  = dataset["test"].map(convert_label)


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

⚖️ Inspect Dataset Balance

In [ ]:
label_counts = Counter(dataset["train"]["label"])
print("\nLabel counts in train:")
print(label_counts)


Label counts in train:
Counter({1: 5362, 0: 4666, 3: 2159, 4: 1937, 2: 1304, 5: 572})


⚖️ Class Weights

In [ ]:
counts = torch.tensor([4666, 5362, 1304, 2159, 1937, 572], dtype=torch.float)
weights = 1.0 / counts
weights = weights / weights.sum()          # normalized weights

print("\nClass weights:", weights)


Class weights: tensor([0.0550, 0.0479, 0.1969, 0.1189, 0.1325, 0.4488])


The dataset is unbalanced.  And so I add class weights (loss weights) so that more rare classes have more weight

🧠 Options & Global Constants

In [ ]:
OPTIONS = ["sadness", "joy", "love", "anger", "fear", "surprise"]
label2id = {label: i for i, label in enumerate(OPTIONS)}

## 🧩 Create Prompts from Tabular Data



In [ ]:
def row_to_prompt(example):
    options_text = "\n".join(f"- {opt}" for opt in OPTIONS)
    prompt = (
        "You are an expert in emotion classification.\n\n"
        f"Text:\n{example['text']}\n\n"
        f"Options:\n{options_text}\n\n"
        "Answer with exactly one option from the list."
    )
    return {
        "prompt": prompt,
        "label_id": label2id[example["emotion"]],
    }

train_ds = dataset["train"].map(row_to_prompt)
test_ds  = dataset["test"].map(row_to_prompt)

print("\nExample prompt row:")
print(train_ds[0])


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


Example prompt row:
{'text': 'i didnt feel humiliated', 'label': 0, 'emotion': 'sadness', 'prompt': 'You are an expert in emotion classification.\n\nText:\ni didnt feel humiliated\n\nOptions:\n- sadness\n- joy\n- love\n- anger\n- fear\n- surprise\n\nAnswer with exactly one option from the list.', 'label_id': 0}


🧪 Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

🧠 Shared Tokenizer

In [ ]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

tok = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token,
    use_fast=True,
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# We'll classify by looking only at these token IDs
answer_token_ids = [
    tok(opt, add_special_tokens=False).input_ids[-1]
    for opt in OPTIONS
]
answer_token_ids_tensor = torch.tensor(answer_token_ids, dtype=torch.long)

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

🔢 Tokenize for Training

In [ ]:
def tokenize_fn(batch):
    enc = tok(
        batch["prompt"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    enc["labels"] = batch["label_id"]  # class index 0..5
    return enc

train_encoded = train_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_ds.column_names,
)
test_encoded = test_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=test_ds.column_names,
)

# Make datasets return torch tensors
train_encoded.set_format(type="torch")
test_encoded.set_format(type="torch")

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

🧠 Baseline Model (Unfine-tuned)

In [ ]:
baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=hf_token,
)
print("\nLoaded baseline_model (unfine-tuned).")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


Loaded baseline_model (unfine-tuned).


📊 Shared Evaluation Function

In [ ]:
def evaluate_model(model, ds_with_prompts, name: str, n_preview: int = 500):
    """
    Evaluate a model using the same "next-token over OPTIONS" trick.
    Uses ds_with_prompts (with 'prompt' + 'label_id').
    """
    model.eval()
    N = min(n_preview, len(ds_with_prompts))

    y_true, y_pred = [], []

    for i in range(N):
        ex = ds_with_prompts[i]
        prompt = ex["prompt"]
        label_id = ex["label_id"]

        inputs = tok(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)
            logits_full = outputs.logits[:, -1, :]                     # [1, vocab]
            option_logits = logits_full[:, answer_token_ids_tensor]    # [1, 6]
            pred_class = option_logits.argmax(dim=-1).item()

        y_true.append(label_id)
        y_pred.append(pred_class)

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    print(f"\n====== {name} (N={N}) ======")
    print(f"Acc={acc:.3f}  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, target_names=OPTIONS, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print("   " + "  ".join(OPTIONS))
    for i, row in enumerate(cm):
        print(f"{OPTIONS[i]:8s} {row}")

    return y_true, y_pred, cm

📊 Baseline Evaluation

In [ ]:
_ = evaluate_model(baseline_model, test_ds, name="Baseline (unfine-tuned)")


====== Baseline (unfine-tuned) (N=500) ======
Acc=0.284  Prec=0.196  Rec=0.279  F1=0.182

Classification report:
              precision    recall  f1-score   support

     sadness       0.00      0.00      0.00       148
         joy       0.80      0.38      0.51       151
        love       0.17      0.44      0.24        39
       anger       0.21      0.86      0.33        79
        fear       0.00      0.00      0.00        70
    surprise       0.00      0.00      0.00        13

    accuracy                           0.28       500
   macro avg       0.20      0.28      0.18       500
weighted avg       0.29      0.28      0.23       500


Confusion matrix (rows=true, cols=pred):
   sadness  joy  love  anger  fear  surprise
sadness  [  0   4  24 120   0   0]
joy      [ 0 57 38 56  0  0]
love     [ 0  7 17 15  0  0]
anger    [ 0  2  9 68  0  0]
fear     [ 0  1  8 61  0  0]
surprise [0 0 5 8 0 0]


⚙️ QLoRA Base Model (4-bit) for Training

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

sft_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)
print("\nLoaded sft_base_model (4-bit, QLoRA-ready).")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_model = get_peft_model(sft_base_model, lora_config)
print("Wrapped sft_base_model with LoRA → sft_model.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


Loaded sft_base_model (4-bit, QLoRA-ready).
Wrapped sft_base_model with LoRA → sft_model.


🛠️ Custom Trainer with loss_fn

In [ ]:
class WeightedLossTrainer(Trainer):
    """
    Custom Trainer that:
    - Uses class-weighted CrossEntropyLoss
    - Only uses logits over the 6 emotion option tokens
    """
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")                             # [batch]
        outputs = model(**inputs)
        logits_full = outputs.logits                              # [batch, seq_len, vocab]
        last_logits = logits_full[:, -1, :]                       # [batch, vocab]
        option_logits = last_logits[:, answer_token_ids_tensor]   # [batch, 6]

        # Put weights and labels are on the same device as logits
        device = option_logits.device
        weighted_ce = CrossEntropyLoss(weight=weights.to(device))
        loss = weighted_ce(option_logits, labels.to(device))

        if return_outputs:
            return loss, outputs
        return loss


🎯 Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="emotion-llama-qlora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    num_train_epochs=1,
    bf16=True,
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    report_to="none",   # disable wandb etc.
)


🚀 Train

In [ ]:
trainer = WeightedLossTrainer(
    model=sft_model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=test_encoded,
    tokenizer=tok,
)

print("\nFinished QLoRA fine-tuning; sft_model is trained.")


Finished QLoRA fine-tuning; sft_model is trained.


/tmp/ipython-input-2143597924.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedLossTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedLossTrainer(


🚀 QLoRA Fine-Tuning

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
50,25.230000
100,18.904500
150,15.000000
200,11.402700
250,9.866000
300,8.024000
350,7.178600
400,6.360700
450,6.036800
500,5.172200


TrainOutput(global_step=1000, training_loss=7.71379574584961, metrics={'train_runtime': 4951.0302, 'train_samples_per_second': 3.232, 'train_steps_per_second': 0.202, 'total_flos': 3.69217064927232e+17, 'train_loss': 7.71379574584961, 'epoch': 1.0})

💾 Save LoRA

In [ ]:
adapter_dir = "emotion-llama-qlora"
sft_model.save_pretrained(adapter_dir)
tok.save_pretrained(adapter_dir)
print(f"\nSaved LoRA adapter + tokenizer to: {adapter_dir}")

sft_model.push_to_hub("david125tran/emotion-llama-qlora")
tok.push_to_hub("david125tran/emotion-llama-qlora")


Saved LoRA adapter + tokenizer to: emotion-llama-qlora


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|4         | 1.12MB / 27.3MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...lama-qlora/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/david125tran/emotion-llama-qlora/commit/48cd639ab17fcbfa2d66ed12e2ce67b4cfae9a5c', commit_message='Upload tokenizer', commit_description='', oid='48cd639ab17fcbfa2d66ed12e2ce67b4cfae9a5c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/david125tran/emotion-llama-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='david125tran/emotion-llama-qlora'), pr_revision=None, pr_num=None)

🔄 Reload Fine-Tuned Model for Evaluation

In [ ]:
reload_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,   # use same 4-bit config
    device_map="auto",
    token=hf_token,
)

sft_model_reloaded = PeftModel.from_pretrained(
    reload_base,
    "david125tran/emotion-llama-qlora",
)
print("\nReloaded sft_model_reloaded from Hub.")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/986 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/27.3M [00:00<?, ?B/s]


Reloaded sft_model_reloaded from Hub.


📊 Evaluate Fine-Tuned

In [ ]:
_ = evaluate_model(sft_model_reloaded, test_ds, name="Supervised Fine-Tuning Model")


====== Supervised Fine-Tuning Model (N=500) ======
Acc=0.886  Prec=0.864  Rec=0.868  F1=0.852

Classification report:
              precision    recall  f1-score   support

     sadness       0.94      0.93      0.93       148
         joy       1.00      0.85      0.92       151
        love       0.55      1.00      0.71        39
       anger       0.88      0.89      0.88        79
        fear       0.92      0.86      0.89        70
    surprise       0.90      0.69      0.78        13

    accuracy                           0.89       500
   macro avg       0.86      0.87      0.85       500
weighted avg       0.91      0.89      0.89       500


Confusion matrix (rows=true, cols=pred):
   sadness  joy  love  anger  fear  surprise
sadness  [137   0   5   4   2   0]
joy      [  0 128  22   1   0   0]
love     [ 0  0 39  0  0  0]
anger    [ 4  0  5 70  0  0]
fear     [ 4  0  0  5 60  1]
surprise [1 0 0 0 3 9]
